<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/chunking_strategies_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub langchain_core langchain_huggingface torch tiktoken datasets chonkie[semantic] chonkie chromadb

In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import login


hf_token = ""

# Colab Secrets (🔑 in the sidebar)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass


login(token=hf_token)

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_1.3"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

In [ ]:
sample_text = train_df.loc[train_df["case_text"].str.len().idxmax(), "case_text"]

print(sample_text)

Structured Chunking

In [ ]:
from chonkie import (
    SemanticChunker,
    RecursiveChunker,
    RecursiveRules,
    RecursiveLevel,
    OverlapRefinery,
)

import re

# ---------- Chunkers ----------
rules = RecursiveRules(levels=[
    RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
    RecursiveLevel(delimiters=["\n"], include_delim="prev"),
    RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    RecursiveLevel(whitespace=True),
])

semantic_chunker = SemanticChunker(
    threshold=0.7,
    chunk_size=400,              # tokens
    similarity_window=3,
    min_sentences_per_chunk=2,
    skip_window=0,
)

recursive_chunker = RecursiveChunker(
    tokenizer="character",
    chunk_size=700,              # characters
    rules=rules,
)

overlap_refinery = OverlapRefinery(
    tokenizer="character",
    context_size=120,            # fixed overlap in characters
    mode="recursive",
    rules=rules,
    method="prefix",
    merge=True,
)


In [18]:
import chromadb
client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

Semantic Chunking

In [ ]:
def clean_text(text):
    # Stop "et al." from being treated as a sentence ending
    text = text.replace("et al.", "et al")
    # Remove image tags
    text = re.sub(r"\[?(?:<[^>]+\.webp>)+(?:-\d+)?\]?", "", text)
    return text

# ---------- Chunk functions----------
def perform_semantic_chunking(text):
    chunks = semantic_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]


def perform_structured_chunking(text):
    chunks = recursive_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]